# ChaCha20



ChaCha is a **Stream Cipher**. That means, instead of scrabling the message directly, it first generates a long stream of random-looking bytes and then mixes the message with that stream.

As per the RFC-8439:

```txt
   ChaCha20 successively calls the ChaCha20 block function, with the
   same key and nonce, and with successively increasing block counter
   parameters.  ChaCha20 then serializes the resulting state by writing
   the numbers in little-endian order, creating a keystream block.
   Concatenating the keystream blocks from the successive blocks forms a
   keystream.  The ChaCha20 function then performs an XOR of this
   keystream with the plaintext.
```

It can be thought of as:

```bash
    Message:     HELLO
    Keystream:   Qx9@#
    Result:      (garbage)
```

The inputs to ChaCha20 are:
1. A 256-bit Key
2. A 96-bit nonce
3. A 32-bit initial counter
4. An arbitrary-length plaintext

The output is an encrypted message or "ciphertext", of the same length.

$\fbox{Note:}$ **Why aren't we allowed to reuse a number used once?**  
    &emsp;This is beacause if we reuse the same key paired with nonce, we get the same keystream thereby allowing attackers to deduce information. Since the message is XOR with the keystream.

### The Key

The key is the only secret in this algorithm. It determines:
- Who can generate the same keystream
- Who can decrypt the message

#### Properties:

- 256 bits
- Must remain secret
- Can be reused across many messages safely

The key is like the identifier of the encryption.

### The Nonce

The nonce ensures that the same key never reproduces the same keystream twice.

#### Properties:

- 96 bits
- Must be unique per key
- Can be public
- Random or sequential

The nonce is similar to the message ID.

### The Counter

ChaCha20 generates keystream in 64-byte chunks. The counter ensures that each chunk gets a different keystream, even within the same message.

## The Algorithm

In [17]:
# Utility
from typing import List
import struct

def rotl32(value: int, shift: int) -> int:
    """Rotate a 32-bit integer left"""
    return ((value << shift) & 0xffffffff) | (value >> (32 - shift))

def print_state(state: List[int], title: str):
    print(f"\n--- {title} ---")
    for i in range(4):
        row = state[i*4:(i+1)*4]
        print(" ".join(f"{x:08x}" for x in row))

### 1. Build the starting table

ChaCha20 starts by building a table of 16 numbers in 4x4 grid.

```bin
[ fixed words     ]   ← always the same
[ key words       ]   ← your secret
[ key words       ]
[ counter + nonce ]
```

|C|C|C|C|
|---|---|---|---|
|**K**|**K**|**K**|**K**|
|**K**|**K**|**K**|**K**|
|**B**|**N**|**N**|**N**|

Where, each cell is a 32 bit word

```bash
C - Constant
K - Key
B - 32-bit Block Counter
N - 96-bit Nonce split across 3 words
```

In [18]:
def build_state(key: bytes, counter: int, nonce: bytes) -> List[int]:
    """
    Builds the initial 4x4 ChaCha20 state
    """

    constants = b"expand 32-byte k"

    state = []
    state += struct.unpack("<4I", constants)
    state += struct.unpack("<8I", key)
    state.append(counter)
    state += struct.unpack("<3I", nonce)

    return list(state)


### 2. Copy the table

ChaCha20 immediately makes a copy of this table where one copy stays unchanged and the other copy gets aggressively scrambled.



### 3. Scramble the table

The algorithm now repeatedly scrambles the numbers in the table 20 times i.e 10 double rounds. Each double round = 1 column round and 1 diagonal round. It acieves this by adding numbers, flipping bits (XOR) and rotating bits in a circle.

#### The Mixing Operation

ChaCha20 repeatedly takes four numbers at a time and mixes them so that:
- Changing one bit affects many others
- Patterns disappear
- Output looks random
This operation is called a quarter round. Each quarter round operates on a fixed group of four words (either a column or diagonal).

#### Rounds

ChaCha20 does:
- Mix columns
- Then mix diagonals
- Then repeat

This happens 20 times. Why 20?
- Fewer rounds are faster but riskier
- 20 gives a huge safety margin
- No known attack breaks full 20 rounds

After these rounds, the table looks completely unrelated to the original inputs.

In [19]:
def quarter_round(state: List[int], a: int, b: int, c: int, d: int):
    """
    ChaCha20 quarter round:
    Operates on four indices of the state
    """

    state[a] = (state[a] + state[b]) & 0xffffffff
    state[d] ^= state[a]
    state[d] = rotl32(state[d], 16)

    state[c] = (state[c] + state[d]) & 0xffffffff
    state[b] ^= state[c]
    state[b] = rotl32(state[b], 12)

    state[a] = (state[a] + state[b]) & 0xffffffff
    state[d] ^= state[a]
    state[d] = rotl32(state[d], 8)

    state[c] = (state[c] + state[d]) & 0xffffffff
    state[b] ^= state[c]
    state[b] = rotl32(state[b], 7)

def chacha20_block(key: bytes, counter: int, nonce: bytes) -> bytes:
    """
    Produces one 64-byte keystream block
    """

    state = build_state(key, counter, nonce)
    working_state = state.copy()

    print_state(state, "Initial State")

    for round_num in range(1, 11):
        # Column rounds
        quarter_round(working_state, 0, 4, 8, 12)
        quarter_round(working_state, 1, 5, 9, 13)
        quarter_round(working_state, 2, 6, 10, 14)
        quarter_round(working_state, 3, 7, 11, 15)

        # Diagonal rounds
        quarter_round(working_state, 0, 5, 10, 15)
        quarter_round(working_state, 1, 6, 11, 12)
        quarter_round(working_state, 2, 7, 8, 13)
        quarter_round(working_state, 3, 4, 9, 14)

        print_state(working_state, f"After Double Round {round_num}")

    # Feed-forward
    for i in range(16):
        working_state[i] = (working_state[i] + state[i]) & 0xffffffff

    print_state(working_state, "After Feed-Forward")

    # Serialize
    keystream = b"".join(struct.pack("<I", word) for word in working_state)

    print("\nKeystream (64 bytes):")
    print(keystream.hex())

    return keystream


### 4. Add the original table back in

ChaCha20 now adds the scrambled table to the original table number by number. This feed-forward step prevents the internal permutation from being reversible.


### 5. Turn the table into bytes

The final table is now converted into bytes and is then produced as 64 bytes of output. These 64 bytes are the pure keystream.


### 6. Encrypt Message

ChaCha20 now combines the keystream and plaintext using XOR byte-by-byte.
$$
\text{Encrypted Data} = \text{Message} \oplus \text{Keystream}
$$


In [20]:
def chacha20_xor(data: bytes, key: bytes, nonce: bytes, counter: int = 1) -> bytes:
    result = b""
    block_counter = counter

    for i in range(0, len(data), 64):
        block = data[i:i+64]
        keystream = chacha20_block(key, block_counter, nonce)

        result_block = bytes(b ^ k for b, k in zip(block, keystream))
        result += result_block
        block_counter += 1

    return result

def chacha20_encrypt(plaintext: bytes, key: bytes, nonce: bytes, counter: int = 1) -> bytes:
    print("\n=== ENCRYPTION ===")
    return chacha20_xor(plaintext, key, nonce, counter)

def chacha20_decrypt(ciphertext: bytes, key: bytes, nonce: bytes, counter: int = 1) -> bytes:
    print("\n=== DECRYPTION ===")
    return chacha20_xor(ciphertext, key, nonce, counter)


### 7. Move to Next Block

If the message is longer than 64 bytes:
- Increase the counter
- Build a new table
- Scramble again
- Produce the next keystream block
- Repeat

In [21]:
key = b"\x00" * 32
nonce = b"\x00" * 12
message = b"Hello ChaCha20! This is a fully traced encryption demo."

print("\nOriginal Message:")
print(message)


Original Message:
b'Hello ChaCha20! This is a fully traced encryption demo.'


In [22]:
encrypted = chacha20_encrypt(message, key, nonce)

print("\nFinal Ciphertext:")
print(encrypted.hex())


=== ENCRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
00000000 00000000 00000000 00000000
00000000 00000000 00000000 00000000
00000001 00000000 00000000 00000000

--- After Double Round 1 ---
d35fd249 1be820a9 d31f88da 8e7f397f
c1bf4b8a feb74a18 a1f753eb f91e808c
578cda2e 042c17b1 010a06a9 49f7792a
2bbe8409 c63023e6 ab8f9939 87d62152

--- After Double Round 2 ---
bc389fa8 b9ba7055 3085e7bd 009ee8cf
0ab5eecd 3111471a 86b484ee 7f517f6d
47323b48 1289b2d8 f81f7778 ad21d4ee
6fb0164a add42d2e 81ed68a2 66145871

--- After Double Round 3 ---
05ef5f24 0a23c120 f0ab6935 e071f61f
e2a62d59 57de135b 25e917b1 97526658
cf306ca2 5dc44a81 e020206c c89e2f10
5d348f46 50e06f03 48e9dd8c 7cb1007c

--- After Double Round 4 ---
ac8a366d 1ea1417c f378dd8d 42858c8d
1258fdc0 aaa2f959 8f0ff2dc 6ba266d5
38ec3250 98dac5bb 566f0cee 652a878b
25bf8a9f bb21eb1d d8e5564b aa681e82

--- After Double Round 5 ---
b42c1d68 2a9852a5 7d93a117 44f144cd
e6eb9a9e 15e413ae ea58f506 6af9e90c
6a11b951 550c05

In [23]:
decrypted = chacha20_decrypt(encrypted, key, nonce)

print("\nDecrypted Message:")
print(decrypted)


=== DECRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
00000000 00000000 00000000 00000000
00000000 00000000 00000000 00000000
00000001 00000000 00000000 00000000

--- After Double Round 1 ---
d35fd249 1be820a9 d31f88da 8e7f397f
c1bf4b8a feb74a18 a1f753eb f91e808c
578cda2e 042c17b1 010a06a9 49f7792a
2bbe8409 c63023e6 ab8f9939 87d62152

--- After Double Round 2 ---
bc389fa8 b9ba7055 3085e7bd 009ee8cf
0ab5eecd 3111471a 86b484ee 7f517f6d
47323b48 1289b2d8 f81f7778 ad21d4ee
6fb0164a add42d2e 81ed68a2 66145871

--- After Double Round 3 ---
05ef5f24 0a23c120 f0ab6935 e071f61f
e2a62d59 57de135b 25e917b1 97526658
cf306ca2 5dc44a81 e020206c c89e2f10
5d348f46 50e06f03 48e9dd8c 7cb1007c

--- After Double Round 4 ---
ac8a366d 1ea1417c f378dd8d 42858c8d
1258fdc0 aaa2f959 8f0ff2dc 6ba266d5
38ec3250 98dac5bb 566f0cee 652a878b
25bf8a9f bb21eb1d d8e5564b aa681e82

--- After Double Round 5 ---
b42c1d68 2a9852a5 7d93a117 44f144cd
e6eb9a9e 15e413ae ea58f506 6af9e90c
6a11b951 550c05

In [24]:
import secrets

key = secrets.token_bytes(32)
nonce = secrets.token_bytes(12)

def hexify(b: bytes) -> str:
    return b.hex()

plaintext_str = input("Enter plaintext: ")
plaintext = plaintext_str.encode("utf-8")

print("\nGenerated Key (hex):")
print(hexify(key))

print("\nGenerated Nonce (hex):")
print(hexify(nonce))


Generated Key (hex):
f9dd8b955d92d767f1d63b8d52129ed6c316334b7a192269be68004328e262f5

Generated Nonce (hex):
7a667d7213b1484ecba452c0


In [25]:
ciphertext = chacha20_encrypt(
    plaintext=plaintext,
    key=key,
    nonce=nonce,
    counter=1
)

print("\nFinal Ciphertext (hex):")
print(hexify(ciphertext))


=== ENCRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
958bddf9 67d7925d 8d3bd6f1 d69e1252
4b3316c3 6922197a 430068be f562e228
00000001 727d667a 4e48b113 c052a4cb

--- After Double Round 1 ---
79b7eab1 a66d1859 5d9b0752 8f684f7d
44c8395b 86240eea 94fc143a 35a0cae2
8e737a42 c5b59df1 85d5954e cc475971
bfabe8b2 a358a8b7 9a77426b 13ed372c

--- After Double Round 2 ---
18cf5265 b5c1ef95 df5b5ede a42ba112
4c4c7ba7 54698cda 39e92467 d486ddac
6bf7d364 9fe0978b 291d99bc bafd738a
8cb321d8 25c09912 e5aca995 a6ee3edb

--- After Double Round 3 ---
061b8de8 1caf1c80 49d32ff5 9288f542
0e90972f cfd774fe 5a04fed8 309cc1a6
2dfb08e6 e2f84b1b a8895bdb 8d4b784a
f890b99a e85cabde 3b80a414 3c2e137e

--- After Double Round 4 ---
ec0c29c9 be1779e8 80529cb7 51939bbc
308cc6f5 223c9815 a29bd1f5 72c8cd33
d7c382ff 1ddae66a 82404bfe 5dd648ee
8a3e8620 dac4d171 c89d16f1 fc6fd61e

--- After Double Round 5 ---
d4ffb0fc 9244f43d 2e0e87e8 b3e77e0b
db2c4bf4 24a2f1b7 1a02d7eb 98327d70
2c498bc4 018e5e

In [26]:
decrypted = chacha20_decrypt(
    ciphertext=ciphertext,
    key=key,
    nonce=nonce,
    counter=1
)

print("\nDecrypted Plaintext:")
print(decrypted.decode("utf-8"))


=== DECRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
958bddf9 67d7925d 8d3bd6f1 d69e1252
4b3316c3 6922197a 430068be f562e228
00000001 727d667a 4e48b113 c052a4cb

--- After Double Round 1 ---
79b7eab1 a66d1859 5d9b0752 8f684f7d
44c8395b 86240eea 94fc143a 35a0cae2
8e737a42 c5b59df1 85d5954e cc475971
bfabe8b2 a358a8b7 9a77426b 13ed372c

--- After Double Round 2 ---
18cf5265 b5c1ef95 df5b5ede a42ba112
4c4c7ba7 54698cda 39e92467 d486ddac
6bf7d364 9fe0978b 291d99bc bafd738a
8cb321d8 25c09912 e5aca995 a6ee3edb

--- After Double Round 3 ---
061b8de8 1caf1c80 49d32ff5 9288f542
0e90972f cfd774fe 5a04fed8 309cc1a6
2dfb08e6 e2f84b1b a8895bdb 8d4b784a
f890b99a e85cabde 3b80a414 3c2e137e

--- After Double Round 4 ---
ec0c29c9 be1779e8 80529cb7 51939bbc
308cc6f5 223c9815 a29bd1f5 72c8cd33
d7c382ff 1ddae66a 82404bfe 5dd648ee
8a3e8620 dac4d171 c89d16f1 fc6fd61e

--- After Double Round 5 ---
d4ffb0fc 9244f43d 2e0e87e8 b3e77e0b
db2c4bf4 24a2f1b7 1a02d7eb 98327d70
2c498bc4 018e5e